# Bibliotecas

## Sistema (os, makedirs, glob)

In [5]:
import os
from os import makedirs
from os import listdir
import glob

## Básicos (numpy, math, display, locale, time, random, re)

In [6]:
# !python -m pip install jupyter


# !python -m pip install IPython
from IPython.display import display

import math

# !python -m pip install numpy
import numpy as np

import locale
# locale.setlocale(locale.LC_ALL, "pt_BR.UTF-8")  # Use "" for auto, or force e.g. to "en_US.UTF-8"

import time
from datetime import datetime, timedelta, date

from pandas.tseries.offsets import BDay # para os dias úteis
# today = datetime.datetime.today()
# print(today - BDay(4)) # 4 dias úteis atrás

# # !python -m pip install random
import random
random.seed(42)

# !python -m pip install regex
import re

## Leitura e análise de dados (Excel, Pandas, Spark)

In [7]:
# !python -m pip install findspark

# !python -m pip install openpyxl
# import openpyxl

# !python -m pip install xlsxwriter
# import xlsxwriter

# !python -m pip install xlrd
# import xlrd

# !python -m pip install python-calamine
# import python_calamine


# !python -m pip install pandas
import pandas as pd
pd.options.display.float_format = '{:,.2f}'.format
# pd.set_option('display.float_format', lambda x: '%.2f' % x)

## Finanças (yfinance, mplfinance)

In [8]:
# https://pypi.org/project/yfinance/
# https://github.com/ranaroussi/yfinance/wiki/Ticker

!python -m pip install yfinance
import yfinance as yf

!python -m pip install mplfinance
import mplfinance as mpf

# # Em R
# # https://cran.r-project.org/web/packages/BatchGetSymbols/index.html

Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable


## Visualização (matplotlib, seaborn, plotly)

In [ ]:
# !python -m pip install matplotlib
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib import ticker
from matplotlib.ticker import FormatStrFormatter, StrMethodFormatter

# !python -m pip install seaborn
import seaborn as sns

# !python -m pip install plotly
# import plotly.graph_objects as go
# import plotly.express as px
# from plotly.subplots import make_subplots

# !python -m pip install graphviz
# import graphviz

Defaulting to user installation because normal site-packages is not writeable


# Funções

### Agrupar cada coluna (_agrupamento_cada_coluna_)

In [10]:
def agrupamento_cada_coluna(
    bd, 
    coluna_completa, 
    colunas_ignoradas = [], 
    ascending = False,
    imprime_tabelas = True,
    ):
    
    from IPython.display import display

    campos_com_erro = []
    dict_campos = {}

    colunas = bd.columns.drop(coluna_completa)

    if len(colunas_ignoradas) > 0:
        colunas = colunas.drop(colunas_ignoradas)
    
    for coluna in colunas:
        try: 
            temp_coluna = bd.fillna("(vazio)").groupby(coluna).count()[[coluna_completa]].rename(columns = {coluna_completa: "Quantidade"}).sort_values("Quantidade", ascending = ascending)

            if ascending == False:
                temp_coluna["%"] = temp_coluna["Quantidade"]/temp_coluna["Quantidade"].sum()
                temp_coluna["% acumulado"] = temp_coluna["%"].cumsum()
            
            if imprime_tabelas == True:
                display(temp_coluna)
                
            dict_campos[coluna] = temp_coluna
            # limpa(temp_coluna)
        
        except:
            campos_com_erro.append(coluna)
            #print("Campo " + coluna + " deu erro =/")
    
    return [campos_com_erro, dict_campos]

### Elimina colunas NA (_elimina_colunas_NA_)

In [11]:
def elimina_colunas_NA(
    base,
    imprime_colunas_vazias = False,
    imprime_colunas_completas = False,
    imprime_colunas_parciais = False,
    remove_da_base = True,
    retorna_parciais = False
    ):

    tamanho_da_base = len(base)

    colunas_vazias = []
    colunas_completas = []
    colunas_parciais = []

    for coluna in base.columns:
        tamanho_da_coluna = len(base[base[coluna].isna()])
        if tamanho_da_coluna == tamanho_da_base:
            # print(coluna)
            colunas_vazias.append(coluna)
        elif tamanho_da_coluna == 0:
            colunas_completas.append(coluna)
        else:
            colunas_parciais.append(coluna)

    if imprime_colunas_vazias == True:
        if len(colunas_vazias) == 0:
            print("Não há colunas NA")
        else:
            print("Colunas vazias: " + str(colunas_vazias))
            
    if imprime_colunas_completas == True:
        if len(colunas_completas) == 0:
            print("Não há colunas completas")
        else:
            print("Colunas completas: " + str(colunas_completas))
    
    if imprime_colunas_parciais == True:
        if len(colunas_parciais) == 0:
            print("Não há colunas parciais")
        else:
            print("Colunas parciais: " + str(colunas_parciais))

    if remove_da_base == True:
        base = base.drop(colunas_vazias, axis = 1)
    
    if retorna_parciais == True:
        return [base, colunas_parciais]
    else:
        return base

### Análise exploratória básica de todos os campos da base - MUITO ÚTIL (_analise_exploratoria_)

In [12]:
def analise_exploratoria(
    bd,
    imprime_todas_colunas = False,
    imprime_info_colunas = True,
    imprime_colunas_vazias = False,
    imprime_colunas_completas = False,
    imprime_colunas_parciais = False,
    remove_da_base = True,
    retorna_parciais = False,
    # detalhar_colunas_parciais = True,
    colunas_ignoradas = []
    ):

    # COMEÇANDO PELAS COLUNAS DISPONÍVEIS E INFO
    if imprime_todas_colunas == True:
        display(bd.columns)
    
    if imprime_info_colunas == True:
        for i in range(int(np.ceil(len(bd.columns)/20))):
            display(bd.iloc[:, (i*20):min((i+1)*20, len(bd.columns))].info())

    # DETALHAMENTO DE QUAIS COLUNAS SÃO NA OU PARCIAIS
    if retorna_parciais == True:
        [bd_semNA, colunas_parciais] = elimina_colunas_NA(
            bd,
            imprime_colunas_vazias = imprime_colunas_vazias,
            imprime_colunas_completas = imprime_colunas_completas,
            imprime_colunas_parciais = imprime_colunas_parciais,
            remove_da_base = remove_da_base,
            retorna_parciais = True
        )

        print(colunas_parciais)

        # if detalhar_colunas_parciais == True:
            # for coluna in colunas_parciais:
                # print(coluna)
                # print("# " + coluna + ": " + str(len(colunas_parciais[colunas_parciais[coluna].isna()])))
    else:
        colunas_parciais = []

        bd_semNA = elimina_colunas_NA(
            bd,
            imprime_colunas_vazias = imprime_colunas_vazias,
            imprime_colunas_completas = imprime_colunas_completas,
            imprime_colunas_parciais = imprime_colunas_parciais,
            remove_da_base = remove_da_base,
            retorna_parciais = retorna_parciais
        )

    [campos_com_erro, colunas_agrupadas] = agrupamento_cada_coluna(
        bd.reset_index(), 
        coluna_completa = bd_semNA.drop(colunas_parciais, axis = 1).reset_index().columns[0],
        colunas_ignoradas = colunas_ignoradas
    )
    
    if retorna_parciais == True:
        return [bd_semNA, colunas_parciais, colunas_agrupadas, campos_com_erro]
    else:
        return [bd_semNA, colunas_agrupadas, campos_com_erro]

## Treinamento de modelos (_treina_e_roda_modelo_)

In [167]:
def treina_e_roda_modelo(
    dados,
    colunas_treino,
    coluna_resultado,
    
    estimador = "SVC",
    proporcao = 0.25,
    SEED = 42,
    print_tamanho = True,
    print_score = True,
    print_confusion_matrix = False,
    
    DummyClassifier__estrategia = None,
    
    DecisionTreeClassifier__retornar_visualizacao = False,
    DecisionTreeClassifier__max_depth = None,
    
    RandomForestClassifier__n_estimators = 100,
    
    feature_selection = None,
    scaler = None,
):
    from sklearn.model_selection import train_test_split
    # from sklearn.preprocessing import StandardScaler
    
    from sklearn.tree import DecisionTreeClassifier
    from sklearn.ensemble import RandomForestClassifier
    # from sklearn.svm import LinearSVC
    from sklearn.svm import SVC
    from sklearn.naive_bayes import MultinomialNB
    from sklearn.dummy import DummyClassifier
    
    # from sklearn.metrics import accuracy_score
    from sklearn.metrics import confusion_matrix
    from sklearn.tree import export_graphviz
    import graphviz
    import os
    os.environ["PATH"] += os.pathsep + 'C:/Program Files/Graphviz/bin' # https://stackoverflow.com/questions/35064304/runtimeerror-make-sure-the-graphviz-executables-are-on-your-systems-path-aft

    np.random.seed(SEED)


    # DIVIDE A BASE NAS PORÇÕES DE TESTE E TREINO, X E Y
    [original_x_treino, original_x_teste, y_treino, y_teste] = train_test_split(
        dados.loc[:, colunas_treino],
        dados.loc[:, coluna_resultado],
        test_size = proporcao,
        # stratify = dados.loc[:, coluna_resultado] # mesma proporção de y
    )
    
    
    # SELECIONA AS FEATURES COM BASE NO ARGUMENTO E REESCALA SE NECESSÁRIO
    if feature_selection is not None:
        feature_selection.fit(original_x_treino, y_treino)
        x_treino = feature_selection.transform(original_x_treino)
        x_teste = feature_selection.transform(original_x_teste)
        
    if (scaler is None) | ((estimador is not None) * (estimador != "DecisionTreeClassifier")): # Não é necessário reescalar para árvore de decisão
        if feature_selection is None:
            x_treino = original_x_treino
            x_teste = original_x_teste
    else:
        if feature_selection is None:
            scaler.fit(original_x_treino)
            x_treino = scaler.transform(original_x_treino)
            x_teste = scaler.transform(original_x_teste)
        else:
            scaler.fit(x_treino)
            x_treino = scaler.transform(x_treino)
            x_teste = scaler.transform(x_teste)


    # IMPRIME O TAMANHO DE CADA PORÇÃO
    if print_tamanho == True:
        print("Tamanho treino: " + str(len(x_treino)))
        print("Tamanho teste: " + str(len(x_teste)))

    

    # INSTANCIA O ESTIMADOR ESCOLHIDO. SE NÃO HOUVER ESTIMADOR, RETORNA APENAS AS PORÇÕES SELECIONADAS E REESCALADAS
    if estimador is None:
        return [
            [original_x_treino, original_x_teste, y_treino, y_teste], 
            [x_treino, x_teste],
            feature_selection
        ]

    elif estimador == "SVC":
        modelo = SVC(gamma = "auto")
    
    elif estimador == "RandomForestClassifier":
        # print(estimador)
        modelo = RandomForestClassifier(n_estimators = RandomForestClassifier__n_estimators)
        print(modelo)

    elif estimador == "DecisionTreeClassifier":
        if DecisionTreeClassifier__max_depth is None:
            modelo = DecisionTreeClassifier()
        else:
            modelo = DecisionTreeClassifier(max_depth = DecisionTreeClassifier__max_depth)

    elif estimador == "MultinomialNB":
        modelo = MultinomialNB()
        
    elif estimador == "Dummy":
        if DummyClassifier__estrategia is None:  
            modelo = DummyClassifier()
        else:
            modelo = DummyClassifier(strategy = DummyClassifier__estrategia)
        
    else:
        return print("Estimador " + estimador + " não encontrado")
    
    
    # TREINA O ESTIMADOR ESCOLHIDO
    try:
        modelo.fit(x_treino, y_treino)
        print(modelo)
    except ValueError:
        return [print("Não foi possível treinar o modelo escolhido"), modelo]

    # AVALIA O MODELO
    
    # previsoes = modelo.predict(x_teste)
    # taxa_de_acerto = accuracy_score(y_teste, previsoes)

    taxa_de_acerto = modelo.score(x_teste, y_teste)
    
    if print_score == True:
        print("Taxa de acerto do modelo " + estimador + ": {:.2%}".format(taxa_de_acerto))
        
    if print_confusion_matrix == True:
        matriz_confusao = confusion_matrix(y_teste, modelo.predict(x_teste))
        # print(matriz_confusao)
        sns.set(font_scale = 2)
        sns.heatmap(matriz_confusao, annot = True, fmt = "d").set(xlabel = "Predição", ylabel = "Real")
        plt.show()
    
    # RETORNA VISÃO GRÁFICA PARA O DECISION TREE CLASSIFIER
    if (DecisionTreeClassifier__retornar_visualizacao == True) * (estimador == "DecisionTreeClassifier"):
        # return export_graphviz(modelo, out_file = None)
        dot_data = export_graphviz(
            modelo, 
            feature_names = colunas_treino,
            filled = True,
            rounded = True,
            class_names = ["Não", "Sim"]
        )
        grafico = graphviz.Source(dot_data)
        return grafico
    
    # RETORNA BASES E MODELO PARA OUTROS ESTIMADORES
    return [
        modelo, 
        [original_x_treino, original_x_teste, y_treino, y_teste], 
        [x_treino, x_teste], 
        feature_selection,
        # previsoes, 
        taxa_de_acerto
    ]

# Leitura dos dados

In [13]:
var_caminho = r"C:\Users\ricardopeloi\OneDrive - falconi365\Data Science\O_Mais_Novo_Day_Trader_do_Brasil\o_mais_novo_day_trader_do_brasil\Bases"
# var_arquivo = r"\Bases\Lista de ações Análise 2025-03-16.xlsx"

lista_arquivos = listdir(var_caminho)
lista_arquivos_analises = []
for arquivo in lista_arquivos:
    if arquivo.find(" Análise") > 0:
        lista_arquivos_analises.append(datetime.strptime(arquivo.split(" Análise ")[1].split(".xls")[0], "%Y-%m-%d"))

var_arquivo_mais_recente = "/Lista de ações Análise " + max(lista_arquivos_analises).strftime("%Y-%m-%d") + ".xlsx"
# print(var_arquivo_mais_recente)

bd_dados_completos = pd.read_excel(var_caminho + var_arquivo_mais_recente).set_index("Ticker")
bd_dados_completos

,Nome da Empresa,Volume no último dia útil (lido em 30/03/2025),industry,sector,fullTimeEmployees,dividendRate,dividendYield,exDividendDate,payoutRatio,beta,...,2025-03-28; Low,2025-03-28; Close,2025-03-28; Volume,2025-03-28; Dividends,2025-03-28; Stock Splits,2025-03-28; HLC,Alfa HLC; últimos 13 dias,Alfa HLC; últimos 55 dias,Martelos,Tipos de Martelos
Ticker,,,,,,,,,,,,,,,,,,,,,
IFCM3,Infracommerce,131606600,Specialty Business Services,Industrials,"2,462.00",NaN,NaN,NaN,0.00,0.54,...,0.08,0.11,131606600,0.00,0,0.10,0.00,0.00,"2025-02-03, 2025-02-04, 2025-02-05, 2025-02-06...","Subida, Subida, Subida, Subida, Subida, Subida..."
HAPV3,Hapvida,68474600,Insurance - Life,Financial Services,NaN,NaN,NaN,"1,640,649,600.00",0.00,0.72,...,2.23,2.30,68474600,0.00,0,2.29,0.01,-0.01,"2025-02-10, 2025-02-13, 2025-02-17, 2025-02-19...","Subida, Subida, Subida, Descida, Subida, Subid..."
COGN3,Cogna,63030500,Education & Training Services,Consumer Defensive,"24,187.00",NaN,NaN,"1,574,294,400.00",0.00,1.13,...,2.01,2.12,63030500,0.00,0,2.09,0.03,0.01,"2025-02-03, 2025-02-04, 2025-02-07, 2025-02-12...","Subida, Descida, Subida, Descida, Descida, Sub..."
CVCB3,CVC,43117200,Travel Services,Consumer Cyclical,NaN,NaN,NaN,"1,577,318,400.00",0.00,NaN,...,2.17,2.26,43119000,0.00,0,2.28,0.04,0.01,"2025-02-10, 2025-02-27, 2025-03-05, 2025-03-12...","Subida, Subida, Subida, Subida, Subida, Descid..."
ITSA4,Itaúsa,40890700,Conglomerates,Industrials,NaN,0.80,8.31,"1,748,822,400.00",0.26,0.69,...,9.55,9.62,40891100,0.00,0,9.63,0.06,0.02,"2025-02-03, 2025-02-10, 2025-02-11, 2025-02-17...","Descida, Subida, Subida, Subida, Descida, Subi..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
BOBR4,Bombril,29000,Household & Personal Products,Consumer Defensive,"2,390.00",NaN,NaN,NaN,0.00,1.10,...,1.68,1.74,29000,0.00,0,1.73,0.00,-0.01,"2025-02-03, 2025-02-04, 2025-02-17, 2025-02-24...","Subida, Descida, Descida, Subida, Subida, Subi..."
RAPT3,Randon,27700,Farm & Heavy Construction Machinery,Industrials,"16,727.00",0.05,0.58,"1,746,144,000.00",0.29,1.11,...,8.04,8.08,27700,0.00,0,8.19,-0.00,0.01,"2025-02-05, 2025-03-10, 2025-03-12, 2025-03-13...","Subida, Subida, Subida, Subida, Descida, Subida,"
OIBR4,Oi,26900,Telecom Services,Communication Services,NaN,NaN,NaN,"1,380,499,200.00",0.00,1.75,...,7.56,7.56,26900,0.00,0,7.78,-0.02,-0.01,"2025-02-11, 2025-02-14, 2025-02-20, 2025-02-21...","Descida, Descida, Descida, Subida, Descida, Su..."


# Análise Exploratória

## Confirmar se coluna Ticker está duplicada

In [14]:
bd_dados_completos[[item for item in bd_dados_completos.columns.to_list() if "Ticker" in item]]

""
Ticker
IFCM3
HAPV3
COGN3
CVCB3
ITSA4
...
BOBR4
RAPT3
OIBR4


## Remover outras colunas que possam se repetir (uma para cada dia)

In [15]:
lista_ignorados_datas = ["Ticker" , "HLC" , "Stock Splits" , "Dividends" , "Volume" , "Close" , "Low" , "High" , "Open"]
# lista_ignorados_datas[0]

lista_colunas_ignoradas_datas = []
for ignorado_datas in lista_ignorados_datas:
    lista_colunas_ignoradas_datas.append([item for item in bd_dados_completos.columns.to_list() if ignorado_datas in item])
# lista_colunas_ignoradas_datas

In [16]:
## Essas colunas já foram removidas no arquivo 'receber_lista_atualizada_tickers.py'

# lista_ignorados = ["industryKey", "industryDisp", "sectorKey", "volume",
#                    "sectorDisp", "regularMarketVolume", "SandP52WeekChange"]

# # lista_colunas_ignoradas_datas = []
# for ignorado_datas in lista_ignorados_datas:
#     lista_ignorados = lista_ignorados + [item for item in bd_dados_completos.columns.to_list() if ignorado_datas in item]
# print(lista_ignorados)


[bd_semNA, colunas_parciais, colunas_agrupadas, campos_com_erro] = analise_exploratoria(
    # bd_dados_completos.drop(lista_ignorados, axis = 1),
    bd_dados_completos,
    imprime_todas_colunas = False,
    # imprime_info_colunas = True,
    imprime_info_colunas = False,
    imprime_colunas_vazias = False,
    imprime_colunas_completas = False,
    imprime_colunas_parciais = False,
    # remove_da_base = True,
    retorna_parciais = True,
    # detalhar_colunas_parciais = True,
    colunas_ignoradas = []
    )

['fullTimeEmployees', 'dividendRate', 'dividendYield', 'exDividendDate', 'payoutRatio', 'beta', 'trailingPE', 'forwardPE', 'priceToSalesTrailing12Months', 'profitMargins', 'trailingEps', 'forwardEps', 'lastSplitFactor', 'lastSplitDate', 'enterpriseToRevenue', 'enterpriseToEbitda', 'lastDividendValue', 'lastDividendDate', 'recommendationMean', 'numberOfAnalystOpinions', 'totalCash', 'totalCashPerShare', 'ebitda', 'totalDebt', 'quickRatio', 'currentRatio', 'totalRevenue', 'debtToEquity', 'revenuePerShare', 'returnOnAssets', 'returnOnEquity', 'grossProfits', 'freeCashflow', 'operatingCashflow', 'earningsGrowth', 'revenueGrowth', 'grossMargins', 'ebitdaMargins', 'epsTrailingTwelveMonths', 'epsForward', 'epsCurrentYear', 'priceEpsCurrentYear']


,Quantidade,%,% acumulado
Nome da Empresa,,,
Banco Santander,3,0.01,0.01
Taesa,3,0.01,0.02
Sanepar,3,0.01,0.04
Klabin,3,0.01,0.05
Eletrobras,2,0.01,0.06
...,...,...,...
Wilson Sons,1,0.00,0.98
Wiz Soluções,1,0.00,0.99
YDUQS,1,0.00,0.99


,Quantidade,%,% acumulado
Volume no último dia útil (lido em 30/03/2025),,,
47000,2,0.01,0.01
172600,2,0.01,0.02
465100,2,0.01,0.02
27700,1,0.00,0.03
24900,1,0.00,0.03
...,...,...,...
40890700,1,0.00,0.98
43117200,1,0.00,0.99
63030500,1,0.00,0.99


,Quantidade,%,% acumulado
industry,,,
Banks - Regional,13,0.05,0.05
Real Estate Services,12,0.05,0.10
Real Estate - Development,11,0.04,0.14
Utilities - Renewable,10,0.04,0.18
Utilities - Regulated Electric,9,0.04,0.22
...,...,...,...
Personal Services,1,0.00,0.98
Specialty Chemicals,1,0.00,0.99
Utilities - Independent Power Producers,1,0.00,0.99


,Quantidade,%,% acumulado
sector,,,
Industrials,49,0.19,0.19
Consumer Cyclical,37,0.15,0.34
Utilities,28,0.11,0.45
Basic Materials,25,0.10,0.55
Real Estate,24,0.10,0.65
Financial Services,23,0.09,0.74
Consumer Defensive,19,0.08,0.81
Healthcare,17,0.07,0.88
Communication Services,11,0.04,0.92


...

,Quantidade,%,% acumulado
2025-03-28; HLC,,,
0.10,1,0.00,0.00
0.25,1,0.00,0.01
0.52,1,0.00,0.01
0.72,1,0.00,0.02
0.90,1,0.00,0.02
...,...,...,...
54.87,1,0.00,0.98
57.83,1,0.00,0.99
67.17,1,0.00,0.99


,Quantidade,%,% acumulado
Alfa HLC; últimos 13 dias,,,
-1.33,1,0.00,0.00
-0.64,1,0.00,0.01
-0.33,1,0.00,0.01
-0.29,1,0.00,0.02
-0.25,1,0.00,0.02
...,...,...,...
0.52,1,0.00,0.98
0.59,1,0.00,0.99
0.60,1,0.00,0.99


,Quantidade,%,% acumulado
Alfa HLC; últimos 55 dias,,,
-0.35,1,0.00,0.00
-0.24,1,0.00,0.01
-0.20,1,0.00,0.01
-0.13,1,0.00,0.02
-0.10,1,0.00,0.02
...,...,...,...
0.17,1,0.00,0.98
0.18,1,0.00,0.99
0.19,1,0.00,0.99


,Quantidade,%,% acumulado
Martelos,,,
"2025-02-03, 2025-02-04, 2025-02-05, 2025-02-06, 2025-02-10, 2025-02-11, 2025-02-12, 2025-02-13, 2025-02-14, 2025-02-18, 2025-02-20, 2025-02-21, 2025-02-24, 2025-02-25, 2025-02-26, 2025-02-27, 2025-02-28, 2025-03-05, 2025-03-06, 2025-03-07, 2025-03-10, 2025-03-11, 2025-03-13, 2025-03-14, 2025-03-17, 2025-03-18, 2025-03-21, 2025-03-24, 2025-03-26,",1,0.00,0.00
"2025-02-03, 2025-02-04, 2025-02-05, 2025-02-07, 2025-02-10, 2025-02-17, 2025-02-18, 2025-02-19, 2025-02-25, 2025-02-27, 2025-02-28, 2025-03-05, 2025-03-07, 2025-03-10, 2025-03-19, 2025-03-20, 2025-03-21, 2025-03-25,",1,0.00,0.01
"2025-02-03, 2025-02-04, 2025-02-05, 2025-02-10, 2025-02-17, 2025-02-18, 2025-02-21, 2025-02-26, 2025-03-20,",1,0.00,0.01
"2025-02-03, 2025-02-04, 2025-02-05, 2025-02-14, 2025-02-17, 2025-02-19, 2025-02-20, 2025-03-10, 2025-03-11, 2025-03-14,",1,0.00,0.02
"2025-02-03, 2025-02-04, 2025-02-05, 2025-02-20, 2025-02-25, 2025-02-27, 2025-03-10, 2025-03-13, 2025-03-17, 2025-03-25, 2025-03-27, 2025-03-28,",1,0.00,0.02
...,...,...,...
"2025-02-26, 2025-03-13,",1,0.00,0.98
"2025-02-27, 2025-02-28, 2025-03-07, 2025-03-18, 2025-03-19, 2025-03-20, 2025-03-25, 2025-03-26,",1,0.00,0.99
"2025-02-27, 2025-03-06, 2025-03-10, 2025-03-11, 2025-03-21,",1,0.00,0.99


,Quantidade,%,% acumulado
Tipos de Martelos,,,
"Subida, Subida, Subida, Subida, Subida,",4,0.02,0.02
"Subida, Subida, Descida, Subida, Subida, Subida,",4,0.02,0.03
"Subida, Subida, Subida, Subida, Descida,",3,0.01,0.04
"Descida, Descida, Subida, Descida, Descida,",2,0.01,0.05
"Descida, Subida, Subida, Descida,",2,0.01,0.06
...,...,...,...
"Subida, Subida, Subida, Subida, Subida, Subida, Subida, Subida, Subida, Descida, Subida, Subida, Subida,",1,0.00,0.98
"Subida, Subida, Subida, Subida, Subida, Subida, Subida, Subida, Subida, Subida,",1,0.00,0.99
"Subida, Subida, Subida, Subida, Subida, Subida, Subida, Subida, Subida, Subida, Subida,",1,0.00,0.99


## Teste de variáveis específicas

In [17]:
display(len(bd_semNA))
display(len(bd_dados_completos))

252

252

In [18]:
colunas_parciais

['fullTimeEmployees',
 'dividendRate',
 'dividendYield',
 'exDividendDate',
 'payoutRatio',
 'beta',
 'trailingPE',
 'forwardPE',
 'priceToSalesTrailing12Months',
 'profitMargins',
 'trailingEps',
 'forwardEps',
 'lastSplitFactor',
 'lastSplitDate',
 'enterpriseToRevenue',
 'enterpriseToEbitda',
 'lastDividendValue',
 'lastDividendDate',
 'recommendationMean',
 'numberOfAnalystOpinions',
 'totalCash',
 'totalCashPerShare',
 'ebitda',
 'totalDebt',
 'quickRatio',
 'currentRatio',
 'totalRevenue',
 'debtToEquity',
 'revenuePerShare',
 'returnOnAssets',
 'returnOnEquity',
 'grossProfits',
 'freeCashflow',
 'operatingCashflow',
 'earningsGrowth',
 'revenueGrowth',
 'grossMargins',
 'ebitdaMargins',
 'epsTrailingTwelveMonths',
 'epsForward',
 'epsCurrentYear',
 'priceEpsCurrentYear']

In [19]:
# colunas_agrupadas.get("52WeekChange")

colunas_agrupadas.get(colunas_parciais[0])

,Quantidade,%,% acumulado
fullTimeEmployees,,,
(vazio),163,0.65,0.65
"6,047.00",3,0.01,0.66
"55,646.00",3,0.01,0.67
"4,396.00",2,0.01,0.68
"16,027.00",2,0.01,0.69
...,...,...,...
"64,616.00",1,0.00,0.98
"87,000.00",1,0.00,0.99
"100,000.00",1,0.00,0.99


In [20]:
campos_com_erro

[]

# Preparação de dados

## One hot encoding

## Balanceamento

## Separação treino e teste

In [ ]:
# train, test = train_test_split(dados, test_size=.3, random_state=7128)

## Normalização

# Primeiro modelo de IA basicão

In [134]:
var_coluna_objetivo = bd_dados_completos[[item for item in bd_dados_completos.columns.to_list() if "; Close" in item]]

var_coluna_objetivo = var_coluna_objetivo.columns[-1]

var_coluna_objetivo

'2025-03-28; Close'

In [141]:
# # SÓ CONSEGUI FAZER FUNCIONAR REMOVENDO O PARÂMETRO stratify

from sklearn.model_selection import train_test_split
dados = bd_dados_completos
colunas_treino = bd_dados_completos.drop(var_coluna_objetivo, axis = 1).columns
coluna_resultado = var_coluna_objetivo
proporcao = 0.25

# display(dados.loc[:, colunas_treino])
# list(dados.loc[:, coluna_resultado].values)
# dados.loc[:, [coluna_resultado]].iloc[:, 0]

[original_x_treino, original_x_teste, y_treino, y_teste] = train_test_split(
    dados.loc[:, colunas_treino],
    dados.loc[:, [coluna_resultado]],
    test_size = proporcao,
    # stratify = dados.loc[:, [coluna_resultado]] # mesma proporção de y
)

# # [original_x_treino, original_x_teste, y_treino, y_teste]
display(len(original_x_treino))
display(len(original_x_teste))
display(len(y_treino))
display(len(y_teste))

189

63

189

63

In [171]:
# [
#     modelo, 
#     [original_x_treino, original_x_teste, y_treino, y_teste], 
#     [x_treino, x_teste], 
#     feature_selection,
#     # previsoes, 
#     taxa_de_acerto
# ]

[
    [original_x_treino, original_x_teste, y_treino, y_teste], 
    [x_treino, x_teste],
    feature_selection
] = treina_e_roda_modelo(
    dados = bd_dados_completos,
    colunas_treino = bd_dados_completos.drop(var_coluna_objetivo, axis = 1).columns,
    coluna_resultado = var_coluna_objetivo,
    
    estimador = None)

var_treina_e_roda_modelo = treina_e_roda_modelo(
    dados = bd_dados_completos,
    colunas_treino = bd_dados_completos.drop(var_coluna_objetivo, axis = 1).columns,
    coluna_resultado = var_coluna_objetivo,
    
    estimador = "DecisionTreeClassifier",
    proporcao = 0.25,
    SEED = 42,
    print_tamanho = True,
    print_score = True,
    print_confusion_matrix = False,
    
    DummyClassifier__estrategia = None,
    
    DecisionTreeClassifier__retornar_visualizacao = False,
    DecisionTreeClassifier__max_depth = None,
    
    RandomForestClassifier__n_estimators = 100,
    
    feature_selection = None,
    scaler = None,
)

# type(var_treina_e_roda_modelo)
var_treina_e_roda_modelo[0]
var_treina_e_roda_modelo[1].fit(x_treino, y_treino)

Tamanho treino: 189
Tamanho teste: 63
Tamanho treino: 189
Tamanho teste: 63
Não foi possível treinar o modelo escolhido


ValueError: could not convert string to float: 'Tegma'